In [ ]:
# ---------------
# Dependencies
# ---------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

import pandas
import numpy
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    classification_report,
    average_precision_score,
)

In [ ]:
# ------------------
# Reproducibility
# ------------------


def set_seed(seed: int):
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


seed = 50
set_seed(seed)

In [ ]:
# ------------------------------------
# Device Setup (Is CUDA available?)
# ------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
# -----------------------------------
# Load Dataset from GDrive (Colab)
# -----------------------------------

from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ---------------
# Load dataset
# ---------------

df = pandas.read_csv("/content/drive/MyDrive/data/dataset.csv")

if "hash" in df.columns:
    df = df.drop(columns=["hash"])

In [ ]:
X = df.drop(columns=["malware"]).values.astype(numpy.float32)
y = df["malware"].values.astype(numpy.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=df["malware"], random_state=42
)

In [ ]:
X_train = torch.tensor(X_train).to(DEVICE)
X_val = torch.tensor(X_val).to(DEVICE)
y_train = torch.tensor(y_train).to(DEVICE)
y_val = torch.tensor(y_val).to(DEVICE)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128)

In [ ]:
# Hyperparameters
VOCAB_SIZE = 307
SEQ_LEN = 100
NUM_CLASSES = 2
EMB_DIM = 128
EPOCHS = 100

In [ ]:
# Discriminator
class TransformerDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.pos_embedding = nn.Embedding(SEQ_LEN, EMB_DIM)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EMB_DIM, nhead=4, dim_feedforward=256, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.fc = nn.Sequential(
            nn.Linear(EMB_DIM, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(128, NUM_CLASSES),
        )

    def forward(self, x):
        batch_size, seq_len = x.size()
        x = x.long()
        positions = (
            torch.arange(0, seq_len, device=x.device)
            .unsqueeze(0)
            .expand(batch_size, seq_len)
        )
        out = self.embedding(x) + self.pos_embedding(positions)
        out = self.transformer(out)
        out = out.mean(dim=1)
        logits = self.fc(out)
        return logits


D = TransformerDiscriminator().to(DEVICE)
optimizer = optim.Adam(D.parameters(), lr=1e-4)

In [ ]:
# Training
history = {"loss": []}

for epoch in range(EPOCHS):
    D.train()
    total_loss = 0

    for real_x, real_y in train_loader:

        optimizer.zero_grad()

        logits = D(real_x)

        loss = F.cross_entropy(
            logits,
            real_y,
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")
    history["loss"].append(total_loss / len(train_loader))

In [ ]:
history_df = pandas.DataFrame(history)
history_df.to_csv(f"transformer_history_{seed}.csv")
torch.save(D.state_dict(), f"transformer_seed_{seed}.pt")

In [ ]:
state_dict = torch.load("transformer_seed_30.pt")
D.load_state_dict(state_dict)
D.eval()

with torch.no_grad():
    logits = D(X_val)
    probs = F.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)

print(classification_report(y_val.cpu().numpy(), preds.cpu().numpy(), digits=4))

print(
    "PR-AUC:", average_precision_score(y_val.cpu().numpy(), probs[:, 1].cpu().numpy())
)

print("ROC-AUC:", roc_auc_score(y_val.cpu().numpy(), probs[:, 1].cpu().numpy()))

training: 10m

seed 10
```

              precision    recall  f1-score   support

           0     0.8375    0.7160    0.7720       324
           1     0.9929    0.9965    0.9947     12839

    accuracy                         0.9896     13163
   macro avg     0.9152    0.8563    0.8834     13163
weighted avg     0.9890    0.9896    0.9892     13163

PR-AUC: 0.9980465524526271
ROC-AUC: 0.9520430613129941
```

seed 30
```
              precision    recall  f1-score   support

           0     0.8333    0.7716    0.8013       324
           1     0.9942    0.9961    0.9952     12839

    accuracy                         0.9906     13163
   macro avg     0.9138    0.8839    0.8982     13163
weighted avg     0.9903    0.9906    0.9904     13163

PR-AUC: 0.9984315765161942
ROC-AUC: 0.9648745767862003
```

seed 20

```
              precision    recall  f1-score   support

           0     0.8551    0.7284    0.7867       324
           1     0.9932    0.9969    0.9950     12839

    accuracy                         0.9903     13163
   macro avg     0.9241    0.8626    0.8908     13163
weighted avg     0.9898    0.9903    0.9899     13163

PR-AUC: 0.9978920844264739
ROC-AUC: 0.9541860544502235
```

seed 40

```
              precision    recall  f1-score   support

           0     0.8076    0.7901    0.7988       324
           1     0.9947    0.9952    0.9950     12839

    accuracy                         0.9902     13163
   macro avg     0.9011    0.8927    0.8969     13163
weighted avg     0.9901    0.9902    0.9901     13163

PR-AUC: 0.9988885377324608
ROC-AUC: 0.9693529985316729
```

seed 50

```
             precision    recall  f1-score   support

           0     0.8545    0.7068    0.7736       324
           1     0.9926    0.9970    0.9948     12839

    accuracy                         0.9898     13163
   macro avg     0.9236    0.8519    0.8842     13163
weighted avg     0.9892    0.9898    0.9893     13163

PR-AUC: 0.9976693049315164
ROC-AUC: 0.9494927203860921
```